# 07 — Adaptive QEM Experiment Plan

This notebook is the experimental control layer for the study **Adaptive Selection of Quantum Error Mitigation Techniques for Noisy Quantum Circuits on IBM Quantum Hardware**.

It reads the available hardware benchmark/calibration data, applies the transparent rule-based adaptive selector, and creates a reproducible execution plan. It does **not** invent hardware results and does not submit jobs unless `RUN_HARDWARE=True` is explicitly enabled.


In [ ]:
from pathlib import Path
import sys, json
import pandas as pd

PROJECT = Path.cwd()
if (PROJECT / 'adaptive_qem.py').exists():
    ROOT = PROJECT
elif (PROJECT / 'Adaptive_QEM_IBM').exists():
    ROOT = PROJECT / 'Adaptive_QEM_IBM'
else:
    ROOT = PROJECT

sys.path.insert(0, str(ROOT))
from adaptive_qem import AdaptiveQEMSelector

print('Project root:', ROOT.resolve())


## 1. Configuration

Keep hardware execution disabled while validating the plan. Before a real run, record the backend name, calibration timestamp, shots, optimization level, and software versions.

In [ ]:
BACKEND_NAME = 'ibm_kingston'
SHOTS = 4096
OPTIMIZATION_LEVEL = 3
SEED_TRANSPILED = 42
RUN_HARDWARE = False

selector = AdaptiveQEMSelector(
    readout_threshold=0.02,
    two_qubit_error_threshold=0.01,
    depth_threshold=50,
    cx_threshold=20,
    high_depth_threshold=100,
    high_cx_threshold=50,
    raw_noise_threshold=0.005,
)

print('Backend:', BACKEND_NAME)
print('Shots:', SHOTS)
print('Hardware execution enabled:', RUN_HARDWARE)


## 2. Load hardware benchmark data

The notebook searches common locations created by Notebook 04. If the file is absent, the notebook stops rather than creating synthetic hardware measurements.

In [ ]:
candidates = [
    ROOT / 'data/hardware/ibm_kingston_hardware_results.csv',
    ROOT / 'data/hardware/hardware_benchmark_results.csv',
    ROOT / 'data/hardware/ibm_kingston_results.csv',
]

hardware_file = next((p for p in candidates if p.exists()), None)
if hardware_file is None:
    raise FileNotFoundError(
        'No IBM hardware results CSV found. Run Notebook 04 with real hardware execution first.'
    )

hardware_df = pd.read_csv(hardware_file)
print('Loaded:', hardware_file)
display(hardware_df.head())


## 3. Identify calibration columns

Different Qiskit/IBM data-export paths can use different column names. The helper below maps common names without changing the underlying measurements.

In [ ]:
def first_existing(df, names, default=None):
    for name in names:
        if name in df.columns:
            return name
    return default

column_map = {
    'qubits': first_existing(hardware_df, ['transpiled_qubits','qubits','width']),
    'depth': first_existing(hardware_df, ['transpiled_depth','depth']),
    'cx_count': first_existing(hardware_df, ['cx_count','cx','transpiled_cx']),
    'cz_count': first_existing(hardware_df, ['cz_count','cz','transpiled_cz']),
    'swap_count': first_existing(hardware_df, ['swap_count','swap','transpiled_swap']),
    'one_qubit_gate_count': first_existing(hardware_df, ['one_qubit_gate_count','one_qubit_gates']),
    'two_qubit_gate_count': first_existing(hardware_df, ['two_qubit_gate_count','two_qubit_gates']),
    'readout_error': first_existing(hardware_df, ['readout_error','avg_readout_error']),
    'one_qubit_error': first_existing(hardware_df, ['one_qubit_error','avg_1q_error']),
    'two_qubit_error': first_existing(hardware_df, ['two_qubit_error','avg_2q_error','avg_cz_error']),
}
pd.DataFrame({'feature': list(column_map), 'source_column': list(column_map.values())})


## 4. Apply the adaptive selector

The current selector is a transparent rule-based baseline. Its thresholds must be validated against experimental data before being presented as an optimized or statistically learned policy.

In [ ]:
def val(row, key, default=0.0):
    col = column_map.get(key)
    if col is None or pd.isna(row[col]):
        return default
    return row[col]

plans = []
for _, row in hardware_df.iterrows():
    cf = {
        'qubits': val(row, 'qubits'),
        'depth': val(row, 'depth'),
        'cx_count': val(row, 'cx_count'),
        'cz_count': val(row, 'cz_count'),
        'swap_count': val(row, 'swap_count'),
        'one_qubit_gate_count': val(row, 'one_qubit_gate_count'),
        'two_qubit_gate_count': val(row, 'two_qubit_gate_count'),
    }
    hf = {
        'readout_error': val(row, 'readout_error'),
        'one_qubit_error': val(row, 'one_qubit_error'),
        'two_qubit_error': val(row, 'two_qubit_error'),
    }
    d = selector.select_from_dict(cf, hf)
    plans.append({
        'circuit': row.get('circuit', row.get('benchmark', f'row_{len(plans)}')),
        'method': d.method,
        'confidence_margin': d.confidence,
        'reason': d.reason,
        'readout_error': hf['readout_error'],
        'two_qubit_error': hf['two_qubit_error'],
        'depth': cf['depth'],
        'cx_count': cf['cx_count'],
        'swap_count': cf['swap_count'],
    })

plan_df = pd.DataFrame(plans)
display(plan_df)


## 5. Experimental execution matrix

The adaptive policy determines which mitigation path is required. For publication-quality validation, retain a **common baseline raw execution for every circuit**, even when mitigation is selected, so that improvement can be measured against the same hardware condition.

In [ ]:
execution_rows = []
for _, r in plan_df.iterrows():
    methods = ['raw']
    if r['method'] in ('readout_mitigation', 'combined'):
        methods.append('readout_mitigation')
    if r['method'] in ('zne', 'combined'):
        methods.append('zne')
    for method in methods:
        execution_rows.append({
            'circuit': r['circuit'],
            'adaptive_selection': r['method'],
            'execution_method': method,
            'shots': SHOTS,
            'backend': BACKEND_NAME,
            'optimization_level': OPTIMIZATION_LEVEL,
        })

execution_df = pd.DataFrame(execution_rows)
display(execution_df)
print('\nPlanned executions:', len(execution_df))


## 6. ZNE overhead expansion

For ZNE, each selected circuit must be executed at multiple noise scales. The recommended starting point is scale factors 1, 3, and 5. This is an experimental design choice, not a universal optimum.

In [ ]:
ZNE_SCALES = [1, 3, 5]
zne_plan = []
for _, r in plan_df.iterrows():
    if r['method'] in ('zne', 'combined'):
        for sf in ZNE_SCALES:
            zne_plan.append({
                'circuit': r['circuit'],
                'scale_factor': sf,
                'shots': SHOTS,
                'backend': BACKEND_NAME,
            })
zne_plan_df = pd.DataFrame(zne_plan)
display(zne_plan_df)


## 7. Save the adaptive experiment manifest

The manifest records exactly what should be executed. Job IDs and measured outcomes are added only after IBM hardware submission.

In [ ]:
out = ROOT / 'data/hardware'
out.mkdir(parents=True, exist_ok=True)

plan_df.to_csv(out / 'adaptive_qem_selection_plan.csv', index=False)
execution_df.to_csv(out / 'adaptive_qem_execution_matrix.csv', index=False)
zne_plan_df.to_csv(out / 'adaptive_qem_zne_plan.csv', index=False)

manifest = {
    'backend': BACKEND_NAME,
    'shots': SHOTS,
    'optimization_level': OPTIMIZATION_LEVEL,
    'seed_transpiler': SEED_TRANSPILED,
    'hardware_execution_enabled': RUN_HARDWARE,
    'zne_scale_factors': ZNE_SCALES,
    'selector': {
        'readout_threshold': selector.readout_threshold,
        'two_qubit_error_threshold': selector.two_qubit_error_threshold,
        'depth_threshold': selector.depth_threshold,
        'cx_threshold': selector.cx_threshold,
    },
}
(out / 'adaptive_qem_experiment_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('Saved adaptive selection plan, execution matrix, ZNE plan, and manifest to', out)


## 8. Publication data requirements

For every actual IBM execution, retain: benchmark name, backend, job ID, calibration timestamp/snapshot, shots, transpiled depth, 1Q/2Q gate counts, selected mitigation method, scale factor (for ZNE), raw counts, mitigated distribution, success probability, TVD, distribution fidelity, execution count, and wall-clock/runtime information where available.

**Do not compare adaptive QEM only against its selected method.** The strongest experimental design includes raw results for all circuits and the relevant non-adaptive mitigation baselines, allowing the adaptive policy to be evaluated on both reliability and overhead.